### ------------------------------------------------------------------------------------------
### Config + imports
### ------------------------------------------------------------------------------------------


In [12]:
# --- Paths (EDIT THESE) ---
# Use full paths; external drive example on macOS: "/Volumes/YourDriveName/cuprite"
BASE_DIR = r"/Volumes/T7 Shield/sphalerite"   # or whatever folder on the drive has your data
OUT_DIR  = r"/Volumes/T7 Shield/sphalerite_output"
MINES_CSV = r"mindat_sphalerite_localities.csv"

# --- Imports ---
import os, re, json, zipfile, tarfile, tempfile, warnings
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import imageio.v2 as imageio

import matplotlib.pyplot as plt
from pyproj import Transformer, CRS
import contextily as ctx

from PIL import Image
from pycocotools import mask as coco_mask

import xarray as xr
from pyproj import CRS

import xarray as xr
from pyproj import CRS

def ensure_cf_crs(ds: xr.Dataset, crs: CRS, grid_mapping_name: str = "spatial_ref") -> xr.Dataset:
    ds = ds.copy()

    # 1) scalar mapping variable
    ds[grid_mapping_name] = xr.DataArray(
        0,
        attrs={
            "spatial_ref": crs.to_wkt(),
            "crs_wkt": crs.to_wkt(),
        },
    )

    # 2) attach mapping via ENCODING (not attrs)
    for v in ds.data_vars:
        if set(["y", "x"]).issubset(ds[v].dims):
            ds[v].encoding["grid_mapping"] = grid_mapping_name
            # if it exists in attrs from previous attempts, remove it
            ds[v].attrs.pop("grid_mapping", None)

    # keep also globally for your manual fallback
    ds.attrs["crs_wkt"] = crs.to_wkt()
    return ds
    
warnings.filterwarnings("ignore")

### ------------------------------------------------------------------------------------------
### Load mines CSV
### ------------------------------------------------------------------------------------------


In [2]:
mines = pd.read_csv(MINES_CSV)
mines.head()
# Split 'Latitude & Longitude' into 'lat' and 'lon'
for idx, row in mines.iterrows():
    try:
        lat, lon = map(float, row["Latitude & Longitude"].split(","))
        mines.at[idx, "lat"] = lat
        mines.at[idx, "lon"] = lon
    except Exception as e:
        mines.at[idx, "lat"] = np.nan
        mines.at[idx, "lon"] = np.nan
    
assert {"Name", "lat", "lon"}.issubset(mines.columns), mines.columns
mines["mine_id"] = mines.get("mine_id", np.arange(1, len(mines) + 1))
mines.head()

,Name,Latitude & Longitude,Type,MineralsStr,Description,References,LocalityURL,Timestamp,StartedAt,FinishedAt,DurationSec,lat,lon,mine_id
0,Max Mine,"46.63214,13.62932",Mine,"Anhydrite, Baryte, Calcite, Celestine, Cerussi...","Located near Kreuth, about 6 km west of Bad Bl...",['https://www.mindat.org/reference.php?id=1620...,https://www.mindat.org/loc-42681.html,2026-01-16T14:38:26,2026-01-16T14:38:20,2026-01-16T14:38:26,5.96,46.63214,13.62932,1
1,Guelph Dolime Quarry,"43.52111,-80.25861",Quarry,"Calcite, Marcasite, 'Petroleumvar. Bitumen', P...",Other names for this quarry in the past have b...,['https://www.mindat.org/reference.php?id=1633...,https://www.mindat.org/loc-220464.html,2026-01-16T14:40:54,2026-01-16T14:40:48,2026-01-16T14:40:54,5.84,43.52111,-80.25861,2
2,Villaret pit,"44.92417,5.77944",NaN,"Azurite, Baryte, 'Bindheimite', Bornite, Boula...","""South of Grenoble on the N85. On the dumps of...",['https://www.mindat.org/reference.php?id=1618...,https://www.mindat.org/loc-16720.html,2026-01-16T14:41:02,2026-01-16T14:40:56,2026-01-16T14:41:02,6.13,44.92417,5.77944,3
3,Silberthalrücken vein,"47° North , 7° East (est.) Estimate based on o...",NaN,"Anglesite, Ankerite, Azurite, Baryte, Brochant...",Sainte Thérèse old mine worked the Silberthalü...,['https://www.mindat.org/reference.php?id=1629...,https://www.mindat.org/loc-106762.html,2026-01-16T14:41:10,2026-01-16T14:41:04,2026-01-16T14:41:10,6.09,NaN,NaN,4
4,La Sanguinède mine,"43.93482,3.59999",Mine,"'Adamite-Olivenite Series', Anglesite, Aragoni...","Another very well-known mine, famous for the b...",['https://www.mindat.org/reference.php?id=1705...,https://www.mindat.org/loc-211565.html,2026-01-16T14:41:18,2026-01-16T14:41:12,2026-01-16T14:41:18,6.50,43.93482,3.59999,5


### ------------------------------------------------------------------------------------------
### Processor class 
### ------------------------------------------------------------------------------------------


In [3]:
#!/usr/bin/env python3
"""
new_enmap.py (SWIR-only)

EnMAP L2A processor backbone:
- Reads reflectance cube
- Parses true wavelengths + VNIR/SWIR detector identity from *-METADATA.XML
- SORTS by wavelength
- KEEPS ONLY SWIR bands (detector == "SWIR")
- Optional quality masking (cloud, haze, etc.) with correct logic
- Writes CRS + spatial dims so rioxarray.clip works
"""

import warnings
warnings.filterwarnings("ignore")

# Ensure PROJ uses a compatible database before rasterio/rioxarray imports
import os
from pathlib import Path
import importlib.util

os.environ.pop("PROJ_LIB", None)
os.environ.pop("PROJ_DATA", None)
spec = importlib.util.find_spec("rasterio")
if spec and spec.origin:
    rasterio_dir = Path(spec.origin).resolve().parent
    rasterio_proj = rasterio_dir / "proj_data"
    if (rasterio_proj / "proj.db").exists():
        os.environ["PROJ_DATA"] = str(rasterio_proj)

import numpy as np
import xarray as xr
import rioxarray
import xml.etree.ElementTree as ET
from collections import OrderedDict

import zipfile
import tarfile
import tempfile
from typing import Optional, List, Dict


class EnMAPProcessor:
    def __init__(
        self,
        base_dir: str,
        save_netcdf: bool = False,
        netcdf_dir: Optional[str] = None,
        load_quality_layers: bool = False,
        apply_cloud_mask: bool = False,
        remove_clouds: bool = True,
        remove_cloud_shadow: bool = True,
        remove_haze: bool = True,
        remove_snow: bool = True,
        remove_cirrus: bool = True,
        remove_pixel_mask: bool = True,
        remove_classes: bool = True,
        # spectral options
        keep_only_swir: bool = True,
        version: str = "v1",
    ):
        self.base_dir = Path(base_dir)

        self.save_netcdf = save_netcdf
        self.netcdf_dir = (self.base_dir / "netcdf_outputs") if netcdf_dir is None else Path(netcdf_dir)
        if self.save_netcdf:
            self.netcdf_dir.mkdir(parents=True, exist_ok=True)

        self.load_quality_layers = load_quality_layers
        self.apply_cloud_mask = apply_cloud_mask

        self.remove_clouds = remove_clouds
        self.remove_cloud_shadow = remove_cloud_shadow
        self.remove_haze = remove_haze
        self.remove_snow = remove_snow
        self.remove_cirrus = remove_cirrus
        self.remove_pixel_mask = remove_pixel_mask
        self.remove_classes = remove_classes
        self.keep_only_swir = keep_only_swir
        self.version = version
    # ---------------------------------------------------------------------
    # Discovery
    # ---------------------------------------------------------------------
    def find_enmap_data_sources(self) -> Dict[str, List[Path]]:
        sources = {"tarballs": [], "extracted_dirs": [], "zip_files": []}
        # Exclude macOS AppleDouble files (._*.tar.gz), which are not real gzip
        sources["tarballs"] = [p for p in self.base_dir.glob("dims*.tar.gz") if not p.name.startswith("._")]

        for item in self.base_dir.iterdir():
            if item.is_dir():
                l2a_dir = item / "ENMAP.HSI.L2A"
                if l2a_dir.exists():
                    sources["extracted_dirs"].append(item)

        sources["zip_files"] = list(self.base_dir.rglob("*.ZIP")) + list(self.base_dir.rglob("*.zip"))
        return sources

    # ---------------------------------------------------------------------
    # Main read
    # ---------------------------------------------------------------------
    def read_enmap_zip(self, zip_path: Path) -> Optional[xr.Dataset]:
        print(f"\nReading: {zip_path.name}")
        try:
            if zipfile.is_zipfile(zip_path):
                with tempfile.TemporaryDirectory(prefix="enmap_temp_") as tmpdir:
                    with zipfile.ZipFile(zip_path, "r") as z:
                        z.extractall(tmpdir)
                    return self._process_enmap_files(Path(tmpdir), zip_path)
            else:
                return self._process_enmap_files(zip_path.parent, zip_path)
        except Exception as e:
            print(f"  ✗ Error: {e}")
            import traceback
            traceback.print_exc()
            return None

    def _process_enmap_files(self, dir_path: Path, source_path: Path) -> Optional[xr.Dataset]:
        spectral_files = list(dir_path.rglob("*SPECTRAL_IMAGE*.TIF")) + list(dir_path.rglob("*SPECTRAL_IMAGE*.tif"))
        if not spectral_files:
            print("  ⚠ No SPECTRAL_IMAGE file found")
            return None

        spectral_file = spectral_files[0]
        print(f"  Found: {spectral_file.name}")

        da = rioxarray.open_rasterio(spectral_file, masked=True)  # (band,y,x)
        n_bands, height, width = da.shape

        # Parse true wavelengths + detector identity from XML (no fake linspace)
        wl_all, vnir, swir, wl_center, fwhm, swir_idx, xml_used = self._parse_metadata_wavelengths(dir_path)



        if len(wl_all) != n_bands:
            raise ValueError(f"Band count mismatch: raster={n_bands}, XML={len(wl_all)} ({xml_used.name})")

        data = da.values.astype(np.float32)  # (band,y,x) native order
        data_full = data  # keep full-spectrum copy for index calculations

        # BSI/MBI (compute before any spectral filtering; uses VNIR+SWIR bands)
        bsi_blue_nm = 490.0
        bsi_red_nm = 665.0
        bsi_nir_nm = 850.0
        bsi_swir1_nm = 1610.0
        bsi_tol_nm = 20.0

        mbi_swir1_nm = 1610.0
        mbi_swir2_nm = 2200.0
        mbi_nir_nm = 850.0
        mbi_tol_nm = 20.0

        wl_full = np.asarray(wl_center, dtype=np.float32)
        blue_idx = int(np.argmin(np.abs(wl_full - bsi_blue_nm)))
        red_idx = int(np.argmin(np.abs(wl_full - bsi_red_nm)))
        nir_idx = int(np.argmin(np.abs(wl_full - bsi_nir_nm)))
        swir1_idx = int(np.argmin(np.abs(wl_full - bsi_swir1_nm)))
        swir2_idx = int(np.argmin(np.abs(wl_full - mbi_swir2_nm)))

        blue_nm = float(wl_full[blue_idx])
        red_nm = float(wl_full[red_idx])
        nir_nm = float(wl_full[nir_idx])
        swir1_nm = float(wl_full[swir1_idx])
        swir2_nm = float(wl_full[swir2_idx])


        # -------------------------------------------------------
        # FULL SPECTRUM MODE (VNIR + SWIR): do NOT slice by swir_idx
        # -------------------------------------------------------
        if self.keep_only_swir:
            data = data[swir_idx, :, :]             
            wl_center = wl_center[swir_idx]
            fwhm = fwhm[swir_idx]

        # Sort ALL bands by wavelength (monotonic axis)
        wl_full = np.asarray(wl_center, dtype=np.float32)
        fwhm_full = np.asarray(fwhm, dtype=np.float32)
        sort_idx = np.argsort(wl_full)

        wl_sorted = wl_full[sort_idx]
        data_sorted = data[sort_idx, :, :].astype(np.float32)
        fwhm_sorted = fwhm_full[sort_idx]

        # Build ds
        data_vars = {
            "reflectance": (
                ["wavelength", "y", "x"],
                data_sorted,
                {
                    "long_name": "Surface Reflectance",
                    "units": "reflectance",
                    'scale_factor': 0.0001,
                    "nodata": da.rio.nodata if da.rio.nodata is not None else -999,
                },
            )
        }


        if (
            abs(blue_nm - bsi_blue_nm) <= bsi_tol_nm
            and abs(red_nm - bsi_red_nm) <= bsi_tol_nm
            and abs(nir_nm - bsi_nir_nm) <= bsi_tol_nm
            and abs(swir1_nm - bsi_swir1_nm) <= bsi_tol_nm
        ):
            blue = data_full[blue_idx, :, :]
            red = data_full[red_idx, :, :]
            nir = data_full[nir_idx, :, :]
            swir1 = data_full[swir1_idx, :, :]
            bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + 1e-12)
            data_vars["bsi"] = (
                ["y", "x"],
                bsi.astype(np.float32),
                {
                    "long_name": "Bare Soil Index",
                    "blue_nm": blue_nm,
                    "red_nm": red_nm,
                    "nir_nm": nir_nm,
                    "swir1_nm": swir1_nm,
                },
            )
        else:
            print("  ⚠ BSI skipped: missing bands near 490/665/850/1610 nm")

        if (
            abs(swir1_nm - mbi_swir1_nm) <= mbi_tol_nm
            and abs(swir2_nm - mbi_swir2_nm) <= mbi_tol_nm
            and abs(nir_nm - mbi_nir_nm) <= mbi_tol_nm
        ):
            swir1 = data_full[swir1_idx, :, :]
            swir2 = data_full[swir2_idx, :, :]
            nir = data_full[nir_idx, :, :]
            mbi = ((swir1 - swir2 - nir) / (swir1 + swir2 + nir + 1e-12)) + 0.5
            data_vars["mbi"] = (
                ["y", "x"],
                mbi.astype(np.float32),
                {
                    "long_name": "Modified Bare Soil Index",
                    "nir_nm": nir_nm,
                    "swir1_nm": swir1_nm,
                    "swir2_nm": swir2_nm,
                },
            )
        else:
            print("  ⚠ MBI skipped: missing bands near 850/1610/2200 nm")

        quality_data = {}
        if self.load_quality_layers or self.apply_cloud_mask:
            quality_data = self._load_quality_layers(spectral_file.parent, da)
            if self.load_quality_layers:
                for layer_name, layer_array in quality_data.items():
                    data_vars[layer_name] = (
                        ["y", "x"],
                        layer_array.astype(np.uint8),
                        {"long_name": f"Quality layer: {layer_name}", "description": self._get_quality_description(layer_name)},
                    )
        # Build detector labels aligned to wl_center: VNIR if in vnir list else SWIR if in swir list
        # Use tolerance matching (floats won't match exactly).
        vnir_arr = np.asarray(vnir, dtype=np.float32)
        swir_arr = np.asarray(swir, dtype=np.float32)
        tol_nm = 0.6

        ds = xr.Dataset(
            data_vars=data_vars,
            coords={
                "wavelength": ("wavelength", wl_sorted, {"units": "nm"}),
                "y": da.y,
                "x": da.x,
            },
            attrs={
                "crs": str(da.rio.crs) if da.rio.crs else None,
                "crs_wkt": da.rio.crs.to_wkt() if da.rio.crs else None,
                "transform": da.rio.transform(),
                "bounds": da.rio.bounds(),
                "source_file": source_path.name,
                "sensor": "EnMAP",
                "processing_level": "L2A",
                "metadata_xml": xml_used.name,
                "n_bands_raw": int(n_bands),
                "height": int(height),
                "width": int(width),
            },
        )

        # Ensure rioxarray sees CRS + spatial dims (prevents MissingCRS on clip)
        if da.rio.crs is None:
            raise ValueError("GeoTIFF has no CRS. Cannot georeference dataset.")
        ds = ds.rio.write_crs(da.rio.crs)
        ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")

        ds, wl_sorted, fwhm_sorted = self.drop_bands(ds, wl_sorted, fwhm_sorted)
        # ---------------------------------------------------------------------
        # Save SRF in pkl using wl_swir and fwhm_swir
        # ---------------------------------------------------------------------
        import pickle

        srf_dict = OrderedDict(zip(wl_sorted, fwhm_sorted))
        srf_pkl_path = self.base_dir / "full_srf.pkl"
        with open(srf_pkl_path, "wb") as f:
            pickle.dump(srf_dict, f)

        # Optional quality mask (pixel-wise)
        if self.apply_cloud_mask and quality_data:
            mask = self._create_quality_mask(quality_data)
            if mask is not None:
                ds = self._apply_mask_to_dataset(ds, mask)

        print(f"  ✓ Loaded: {ds.sizes['wavelength']} bands (SWIR-only={self.keep_only_swir}), {height}x{width}")
        print(f"  ✓ Wavelength range: {float(ds.wavelength.min()):.1f} - {float(ds.wavelength.max()):.1f} nm")

        if self.save_netcdf:
            self._save_as_netcdf(ds, source_path)

        return ds

    # ---------------------------------------------------------------------
    # XML parsing
    # ---------------------------------------------------------------------
    import numpy as np
    import xml.etree.ElementTree as ET

    def _parse_metadata_wavelengths(self, dir_path: Path):
        import re
        from collections import defaultdict

        xml_files = list(dir_path.rglob("*-METADATA.XML")) + list(dir_path.rglob("*-METADATA.xml"))
        if not xml_files:
            raise FileNotFoundError("No *-METADATA.XML found (no fabricated wavelengths allowed).")

        xml_path = xml_files[0]

        # ---- your existing agg-based extraction (keep it) ----
        float_re = re.compile(r"[-+]?\d*\.\d+|\d+")
        def floats_from_string(s):
            if not s:
                return []
            vals = [float(v) for v in float_re.findall(s)]
            return [v for v in vals if 350 <= v <= 2550]

        path_stack = []
        agg = defaultdict(list)
        for event, elem in ET.iterparse(xml_path, events=("start", "end")):
            if event == "start":
                tag = elem.tag.split("}")[-1]
                path_stack.append(tag)
            else:
                path = "/".join(path_stack)
                agg[path].extend(floats_from_string((elem.text or "").strip()))
                for _, v in elem.attrib.items():
                    agg[path].extend(floats_from_string(str(v)))
                path_stack.pop()
                elem.clear()

        wl_all_path = "level_X/specific/bandCharacterisation/bandID/wavelengthCenterOfBand"
        vnir_path   = "level_X/product/smileCorrection/VNIR/bandID/wavelength"
        swir_path   = "level_X/product/smileCorrection/SWIR/bandID/wavelength"

        wl_all = np.array(agg.get(wl_all_path, []), dtype=float)
        vnir   = np.array(agg.get(vnir_path, []), dtype=float)
        swir   = np.array(agg.get(swir_path, []), dtype=float)

        if len(wl_all) == 0 or len(swir) == 0:
            raise ValueError(f"Failed wavelength parse from {xml_path.name}: wl_all={len(wl_all)}, swir={len(swir)}")

        # ------------------------------------------------------------------
        # NEW: extract FWHM aligned to wl_all using the same XML
        # ------------------------------------------------------------------
        # Parse again (simple) to get bandCharacterisation lists in order
        tree = ET.parse(xml_path)
        root = tree.getroot()

        def strip_ns(tag):  # namespace-safe
            return tag.split("}")[-1]

        wl_bc = []
        fwhm_bc = []

        for band in root.iter():
            if strip_ns(band.tag) != "bandID":
                continue

            # only keep bandID elements that are under bandCharacterisation
            # quick filter: check ancestors is hard in ElementTree; instead check children presence
            w = None
            fw = None
            for child in band:
                t = strip_ns(child.tag)
                if t == "wavelengthCenterOfBand" and child.text:
                    w = float(child.text)
                elif t in {"fwhmOfBand", "FWHMOfBand", "FWHM"} and child.text:
                    fw = float(child.text)

            if w is not None:
                wl_bc.append(w)
                fwhm_bc.append(fw if fw is not None else np.nan)

        wl_bc = np.array(wl_bc, dtype=np.float32)
        fwhm_bc = np.array(fwhm_bc, dtype=np.float32)

        # Sanity: wl_bc should match wl_all length (usually 224)
        if len(wl_bc) != len(wl_all):
            # fall back: you can still proceed, but warn loudly
            print(f"⚠ bandCharacterisation wl length {len(wl_bc)} != wl_all length {len(wl_all)} (XML quirks).")

        # ------------------------------------------------------------------
        # NEW: SWIR indices via tolerance matching (NOT exact float isin)
        # ------------------------------------------------------------------
        # map each swir wavelength to nearest center wavelength
        tol_nm = 0.6  # conservative
        swir_idx = []
        for w in swir.astype(np.float32):
            j = int(np.argmin(np.abs(wl_bc - w)))
            if abs(wl_bc[j] - w) > tol_nm:
                raise ValueError(f"SWIR wavelength {w:.2f} nm not matched to center list (nearest {wl_bc[j]:.2f}).")
            swir_idx.append(j)

        swir_idx = np.array(sorted(set(swir_idx)), dtype=int)

        # Return wl_all etc plus fwhm + swir indices (you choose what to return)
        return wl_all, vnir, swir, wl_bc, fwhm_bc, swir_idx, xml_path

    # ---------------------------------------------------------------------
    # Quality layers + mask (FIXED LOGIC)
    # ---------------------------------------------------------------------
    def _load_quality_layers(self, qdir: Path, reference_da) -> Dict[str, np.ndarray]:
        quality_patterns = {
            "classes": "*QL_QUALITY_CLASSES.TIF",
            "cloud": "*QL_QUALITY_CLOUD.TIF",
            "cloud_shadow": "*QL_QUALITY_CLOUDSHADOW.TIF",
            "haze": "*QL_QUALITY_HAZE.TIF",
            "cirrus": "*QL_QUALITY_CIRRUS.TIF",
            "snow": "*QL_QUALITY_SNOW.TIF",
            "pixel_mask": "*QL_PIXELMASK.TIF",
        }

        out: Dict[str, np.ndarray] = {}
        for name, pattern in quality_patterns.items():
            files = list(qdir.glob(pattern))
            if not files:
                continue
            layer_da = rioxarray.open_rasterio(files[0])

            if (
                (layer_da.rio.crs != reference_da.rio.crs)
                or (layer_da.rio.transform() != reference_da.rio.transform())
                or (layer_da.sizes.get("y") != reference_da.sizes["y"])
                or (layer_da.sizes.get("x") != reference_da.sizes["x"])
            ):
                layer_da = layer_da.rio.reproject_match(reference_da, resampling="nearest")

            arr = layer_da.values
            if arr.ndim == 3 and arr.shape[0] == 1:
                arr = arr[0]
            elif arr.ndim == 3:
                arr = (arr != 0).any(axis=0).astype(np.uint8)

            nodata = layer_da.rio.nodata
            if nodata is not None:
                arr = np.where(arr == nodata, 0, arr)

            out[name] = arr.astype(np.uint8)
            print(f"  ✓ Loaded {name} layer")

        return out

    def _get_quality_description(self, layer_name: str) -> str:
        return {
            "cloud": "0=Clear, 1=Cloud",
            "cloud_shadow": "0=Clear, 1=Shadow",
            "haze": "0=Clear, 1=Haze",
            "snow": "0=Clear, 1=Snow",
            "cirrus": "0=Clear, 1=Cirrus",
            "pixel_mask": "0=Clear, 1=Flagged",
            "classes": "0=Clear, >0=Flagged",
        }.get(layer_name, "Quality layer")

    def _create_quality_mask(self, quality_data: Dict[str, np.ndarray]) -> Optional[np.ndarray]:
        if not quality_data:
            return None

        # True = keep
        sample = next(iter(quality_data.values()))
        H, W = sample.shape
        mask = np.ones((H, W), dtype=bool)

        if self.remove_clouds and "cloud" in quality_data:
            mask &= (quality_data["cloud"] == 0)
        if self.remove_cloud_shadow and "cloud_shadow" in quality_data:
            mask &= (quality_data["cloud_shadow"] == 0)
        if self.remove_haze and "haze" in quality_data:
            mask &= (quality_data["haze"] == 0)
        if self.remove_snow and "snow" in quality_data:
            mask &= (quality_data["snow"] == 0)
        if self.remove_cirrus and "cirrus" in quality_data:
            mask &= (quality_data["cirrus"] == 0)

        # FIXED: keep clear
        if self.remove_pixel_mask and "pixel_mask" in quality_data:
            mask &= (quality_data["pixel_mask"] == 1)

        # FIXED: keep clear
        if self.remove_classes and "classes" in quality_data:
            mask &= (quality_data["classes"] == 1)

        return mask

    def _apply_mask_to_dataset(self, ds: xr.Dataset, mask: np.ndarray) -> xr.Dataset:
        mask_da = xr.DataArray(mask, dims=("y", "x"), coords={"y": ds.y, "x": ds.x})
        ds["reflectance"] = ds["reflectance"].where(mask_da)
        return ds

    # ---------------------------------------------------------------------
    # Save
    # ---------------------------------------------------------------------
    def _save_as_netcdf(self, ds: xr.Dataset, source_path: Path) -> None:
        filename = source_path.stem
        parts = filename.split("_")
        product_id = "_".join(parts[:-2]) if len(parts) >= 3 else filename
        out = self.netcdf_dir / f"{product_id}.nc"

        encoding = {
            "reflectance": {
                "zlib": True,
                "complevel": 4,
                "dtype": "float32",
                "chunksizes": (min(10, ds.sizes["wavelength"]), min(256, ds.sizes["y"]), min(256, ds.sizes["x"])),
            }
        }
        ds.to_netcdf(out, engine="netcdf4", encoding=encoding)
        print(f"  ✓ Saved: {out.name}")

    # ---------------------------------------------------------------------
    # Batch processing (unchanged)
    # ---------------------------------------------------------------------
    def process_extracted_directory(self, dir_path: Path) -> List[xr.Dataset]:
        l2a_dir = dir_path / "ENMAP.HSI.L2A"
        if not l2a_dir.exists():
            print("  ⚠ No ENMAP.HSI.L2A directory found")
            return []
        zip_files = sorted(l2a_dir.glob("*.ZIP"))
        datasets = []
        for z in zip_files:
            ds = self.read_enmap_zip(z)
            if ds is not None:
                datasets.append(ds)
        return datasets

    def process_all_available_data(self) -> Dict[str, List[xr.Dataset]]:
        sources = self.find_enmap_data_sources()
        all_datasets: Dict[str, List[xr.Dataset]] = {}

        for extracted_dir in sorted(sources["extracted_dirs"]):
            dsets = self.process_extracted_directory(extracted_dir)
            if dsets:
                all_datasets[extracted_dir.name] = dsets

        for tarball in sorted(sources["tarballs"]):
            extracted_name = tarball.stem.replace(".tar", "")
            if extracted_name in [d.name for d in sources["extracted_dirs"]]:
                continue
            # Detect format: gzip magic \x1f\x8b; otherwise try uncompressed tar
            try:
                with open(tarball, "rb") as f:
                    magic = f.read(2)
                mode = "r:gz" if magic == b"\x1f\x8b" else "r"
            except Exception:
                mode = "r:gz"
            try:
                with tarfile.open(tarball, mode) as tar:
                    tar.extractall(path=self.base_dir)
                    members = tar.getmembers()
                    if members:
                        root_dir = Path(members[0].name).parts[0]
                        extracted_path = self.base_dir / root_dir
                        dsets = self.process_extracted_directory(extracted_path)
                        if dsets:
                            all_datasets[extracted_path.name] = dsets
            except tarfile.ReadError as e:
                print(f"  ⚠ Skipping {tarball.name}: not a valid tar/gzip file ({e})")

        for zip_file in sorted(sources["zip_files"]):
            ds = self.read_enmap_zip(zip_file)
            if ds is not None:
                all_datasets.setdefault(zip_file.parent.name, []).append(ds)

        return all_datasets

    def drop_bands(
        self,
        ds: xr.Dataset,
        wavelength: Optional[np.ndarray] = None,
        fwhm: Optional[np.ndarray] = None,
    ) -> tuple[xr.Dataset, Optional[np.ndarray], Optional[np.ndarray]]:
        """
        Drop known bad bands for 224-band EnMAP products.
        Also remove the same indices from wavelength and fwhm if provided.
        """
        w = ds["wavelength"].values
        if self.version == "128":
            drop_mask = ((w < 1449.43) & (w > 1330.86))
        elif self.version == "58":
            drop_mask = (w < 1967.66)
        else:
            drop_mask = ~np.ones_like(w, dtype=bool)
        drop_idx = np.where(drop_mask)[0].tolist()
        if drop_idx:
            ds = ds.drop_isel(wavelength=drop_idx)
            if wavelength is not None and len(wavelength) == len(w):
                wavelength = np.delete(wavelength, drop_idx)
            if fwhm is not None and len(fwhm) == len(w):
                fwhm = np.delete(fwhm, drop_idx)
        return ds, wavelength, fwhm
        

### ------------------------------------------------------------------------------------------
### Utility helpers
### ------------------------------------------------------------------------------------------


In [4]:
def safe_name(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s[:80] if len(s) > 80 else s

def save_rgb_png(ds: xr.Dataset, out_png: Path, stretch_p=(2,98), scale=1e-4):
    w = ds["wavelength"].values.astype(float)

    def nearest(target):
        return int(np.argmin(np.abs(w - target)))

    # SWIR false-color (900-2500 nm): R=2200, G=1650, B=1200 nm
    r_i, g_i, b_i = nearest(2200), nearest(1650), nearest(1200)
    R = ds["reflectance"].isel(wavelength=r_i).values * scale
    G = ds["reflectance"].isel(wavelength=g_i).values * scale
    B = ds["reflectance"].isel(wavelength=b_i).values * scale

    rgb = np.stack([R,G,B], axis=-1)
    lo = np.nanpercentile(rgb, stretch_p[0], axis=(0,1))
    hi = np.nanpercentile(rgb, stretch_p[1], axis=(0,1))
    rgb = (rgb - lo) / (hi - lo + 1e-12)
    rgb = np.clip(rgb, 0, 1)

    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.imsave(out_png.as_posix(), rgb)

### ------------------------------------------------------------------------------------------
### Clip functions
### ------------------------------------------------------------------------------------------


In [5]:
def clip_buffer_save(ds: xr.Dataset, lat: float, lon: float, buffer_m: float, out_nc: Path, out_png: Path):
    if ds.rio.crs is None:
        raise ValueError("Dataset has no CRS. Can't clip.")

    transformer = Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)
    x, y = transformer.transform(lon, lat)

    minx, miny, maxx, maxy = ds.rio.bounds()
    if not (minx <= x <= maxx and miny <= y <= maxy):
        return None

    bbox = {
        "type": "Polygon",
        "coordinates": [[
            [x - buffer_m, y - buffer_m],
            [x + buffer_m, y - buffer_m],
            [x + buffer_m, y + buffer_m],
            [x - buffer_m, y + buffer_m],
            [x - buffer_m, y - buffer_m]
        ]]
    }

    clipped = ds.rio.clip([bbox], ds.rio.crs, drop=True)
    if clipped is None:
        return None

    # Materialize so dask doesn't fail mid-write; ensure CRS/transform in file
    clipped = clipped.load()
    clipped = clipped.rio.write_crs(clipped.rio.crs)

    out_nc.parent.mkdir(parents=True, exist_ok=True)
    # Atomic write: temp file then rename (avoids truncated .nc on failure)
    fd, tmp_nc = tempfile.mkstemp(suffix=".nc", dir=out_nc.parent, prefix=".tmp_")
    try:
        os.close(fd)
        clipped.to_netcdf(tmp_nc, engine="netcdf4", format="NETCDF4")
        os.replace(tmp_nc, out_nc)
    except Exception:
        if os.path.exists(tmp_nc):
            try:
                os.unlink(tmp_nc)
            except OSError:
                pass
        raise
    save_rgb_png(clipped, out_png)

    return clipped


def clip_chip_save(
    ds: xr.Dataset,
    lat: float,
    lon: float,
    chip_px: int,
    out_nc: Path,
    out_png: Path,
    x_dim: str = "x",
    y_dim: str = "y",
):
    if ds.rio.crs is None:
        raise ValueError("Dataset has no CRS. Can't clip.")
    if chip_px % 2 != 0:
        raise ValueError("chip_px must be even (e.g. 128).")

    transformer = Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)
    x0, y0 = transformer.transform(lon, lat)

    minx, miny, maxx, maxy = ds.rio.bounds()
    if not (minx <= x0 <= maxx and miny <= y0 <= maxy):
        return None

    xs = ds[x_dim].values
    ys = ds[y_dim].values
    ix = int((abs(xs - x0)).argmin())
    iy = int((abs(ys - y0)).argmin())

    half = chip_px // 2
    x0_i, x1_i = ix - half, ix + half
    y0_i, y1_i = iy - half, iy + half

    if x0_i < 0 or y0_i < 0 or x1_i > xs.shape[0] or y1_i > ys.shape[0]:
        return None

    clipped = ds.isel({x_dim: slice(x0_i, x1_i), y_dim: slice(y0_i, y1_i)})

    if clipped.sizes[x_dim] != chip_px or clipped.sizes[y_dim] != chip_px:
        return None
    
    # --- ensure spatial metadata survives ---
    clipped = clipped.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim, inplace=False)

    # get CRS from ds (or from ds.attrs["crs_wkt"])
    crs = ds.rio.crs
    if crs is None:
        wkt = ds.attrs.get("crs_wkt")
        if not wkt:
            raise ValueError("No CRS in ds.rio.crs or ds.attrs['crs_wkt']")
        crs = CRS.from_wkt(wkt)

    # Write CRS in-memory AND as CF grid_mapping for persistence
    clipped = clipped.rio.write_crs(crs, inplace=False)
    clipped = ensure_cf_crs(clipped, crs)
    
    out_nc.parent.mkdir(parents=True, exist_ok=True)
    encoding = {
            "reflectance": {
                "zlib": True,
                "complevel": 4,
                "dtype": "float32",
                "chunksizes": (min(10, clipped.sizes["wavelength"]), min(256, clipped.sizes["y"]), min(256, clipped.sizes["x"])),
            }
    }
    clipped.to_netcdf(out_nc, engine="netcdf4", format="NETCDF4", encoding=encoding)
    save_rgb_png(clipped, out_png)

    return clipped

### ------------------------------------------------------------------------------------------
### COCO → mask PNG export
### ------------------------------------------------------------------------------------------


In [6]:
# coco_path = Path("_annotations.coco.json")
# out_dir = Path("masks_png")
# out_dir.mkdir(exist_ok=True)

# data = json.loads(coco_path.read_text())

# anns_by_img = {}
# for ann in data["annotations"]:
#     anns_by_img.setdefault(ann["image_id"], []).append(ann)

# cat_to_val = {1: 1}

# for img in data["images"]:
#     img_id = img["id"]
#     H, W = img["height"], img["width"]
#     mask = np.zeros((H, W), dtype=np.uint8)

#     for ann in anns_by_img.get(img_id, []):
#         seg = ann["segmentation"]
#         rles = coco_mask.frPyObjects(seg, H, W)
#         rle = coco_mask.merge(rles) if isinstance(rles, list) else rles
#         m = coco_mask.decode(rle).astype(bool)

#         val = cat_to_val.get(ann["category_id"], 0)
#         mask[m] = val

#     stem = Path(img["file_name"]).stem
#     Image.fromarray(mask).save(out_dir / f"{stem}_mask.png")

# print(f"Saved {len(data['images'])} masks to: {out_dir.resolve()}")

### ------------------------------------------------------------------------------------------
### Main run
### ------------------------------------------------------------------------------------------


In [7]:
processor = EnMAPProcessor(
    base_dir=str(BASE_DIR),   # use BASE_DIR from config (e.g. path on external drive)
    save_netcdf=True,
    load_quality_layers=True,
    apply_cloud_mask=True,
    version=None,
    
)

all_sets = processor.process_all_available_data()
print({k: len(v) for k, v in all_sets.items()})


Reading: ._ENMAP01-____L2A-DT0000001702_20220717T110358Z_005_V010505_20260205T022527Z.ZIP
  ⚠ No SPECTRAL_IMAGE file found

Reading: ._ENMAP01-____L2A-DT0000020106_20230525T105309Z_022_V010505_20260205T022344Z.ZIP
  ⚠ No SPECTRAL_IMAGE file found

Reading: ._ENMAP01-____L2A-DT0000089165_20240823T111649Z_027_V010505_20260205T022104Z.ZIP
  ⚠ No SPECTRAL_IMAGE file found

Reading: ._ENMAP01-____L2A-DT0000141396_20250715T170007Z_007_V010505_20260205T021839Z.ZIP
  ⚠ No SPECTRAL_IMAGE file found

Reading: ._ENMAP01-____L2A-DT0000143736_20250725T162057Z_016_V010505_20260205T023117Z.ZIP
  ⚠ No SPECTRAL_IMAGE file found

Reading: ENMAP01-____L2A-DT0000001702_20220717T110358Z_005_V010505_20260205T022527Z.ZIP
  Found: ENMAP01-____L2A-DT0000001702_20220717T110358Z_005_V010505_20260205T022527Z-SPECTRAL_IMAGE.TIF
  ✓ Loaded classes layer
  ✓ Loaded cloud layer
  ✓ Loaded cloud_shadow layer
  ✓ Loaded haze layer
  ✓ Loaded cirrus layer
  ✓ Loaded snow layer
  ✓ Loaded pixel_mask layer
  ✓ Loaded: 13

In [8]:
out_root = Path(OUT_DIR)
verbose = False
nc_files = list((Path(BASE_DIR)).glob("*/ENMAP*.nc"))
nc_files

[PosixPath('/Volumes/T7 Shield/sphalerite/netcdf_outputs/ENMAP01-____L2A-DT0000001702_20220717T110358Z_005.nc'),
 PosixPath('/Volumes/T7 Shield/sphalerite/netcdf_outputs/ENMAP01-____L2A-DT0000020106_20230525T105309Z_022.nc'),
 PosixPath('/Volumes/T7 Shield/sphalerite/netcdf_outputs/ENMAP01-____L2A-DT0000089165_20240823T111649Z_027.nc'),
 PosixPath('/Volumes/T7 Shield/sphalerite/netcdf_outputs/ENMAP01-____L2A-DT0000141396_20250715T170007Z_007.nc'),
 PosixPath('/Volumes/T7 Shield/sphalerite/netcdf_outputs/ENMAP01-____L2A-DT0000143736_20250725T162057Z_016.nc')]

In [9]:
import xarray as xr
ds = xr.open_dataset('/Volumes/T7 Shield/sphalerite/netcdf_outputs/ENMAP01-____L2A-DT0000001702_20220717T110358Z_005.nc', engine="netcdf4")

In [10]:
ds.rio.crs
wkt = ds.attrs.get("crs_wkt") or (ds["reflectance"].attrs.get("crs_wkt") if "reflectance" in ds else None)
wkt
crs = CRS.from_wkt(wkt)
crs

<Projected CRS: EPSG:32632>
Name: WGS 84 / UTM zone 32N
Axis Info [cartesian]:
- [east]: Easting (metre)
- [north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: UTM zone 32N
- method: Transverse Mercator
Datum: World Geodetic System 1984
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [13]:
datasets_list = []
for nc_file in nc_files:
    ds = xr.open_dataset(nc_file, engine="netcdf4")
    datasets_list.append(ds)


for ds in datasets_list:
    # Ensure dataset has a valid CRS (NetCDF may store it in crs_wkt only)
    crs = getattr(ds.rio, "crs", None)
    if crs is None or not str(crs).strip():
        wkt = ds.attrs.get("crs_wkt") or (ds["reflectance"].attrs.get("crs_wkt") if "reflectance" in ds else None)
        if wkt:
            try:
                crs = CRS.from_wkt(wkt)
                ds = ds.rio.write_crs(crs)
            except Exception:
                crs = None
    if crs is None:
        print("Skipping dataset: no valid CRS (set crs_wkt in attrs or use rioxarray-written NetCDF).")
        continue

    if verbose:
        band_index = 100
        ax = ds["reflectance"].isel(wavelength=band_index).plot(cmap="viridis", figsize=(10, 8))
        plt.title(f"Band {band_index}")
        if ds.rio.crs:
            ax.axes.set_aspect('equal')
            ctx.add_basemap(ax.axes, crs=ds.rio.crs, source=ctx.providers.OpenStreetMap.Mapnik)
        plt.show()

    valid_mines = []
    transformer = Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)
    minx, miny, maxx, maxy = ds.rio.bounds()

    for idx, row in mines.iterrows():
        mine_name = safe_name(row["Name"])
        lat, lon = float(row["lat"]), float(row["lon"])
        x, y = transformer.transform(lon, lat)
        if minx <= x <= maxx and miny <= y <= maxy:
            valid_mines.append((idx, mine_name, lat, lon))

    for idx, mine_name, lat, lon in valid_mines:
        src = Path(ds.attrs.get("source_file", "unknown")).stem

        out_nc  = out_root / "Cuprite Mines" / mine_name / f"{mine_name}_{src}.nc"
        out_png = out_root / "Cuprite Mines" / mine_name / f"{mine_name}_{src}_rgb.png"

        clipped = clip_chip_save(ds=ds, lat=lat, lon=lon, chip_px=128, out_nc=out_nc, out_png=out_png)

        print(f"{mine_name}: clipped None? {clipped is None}")
        print(" NetCDF:", out_nc)
        print(" PNG  :", out_png)

Bschhof_Quarry_Schretzmair_Quarry: clipped None? False
 NetCDF: /Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Bschhof_Quarry_Schretzmair_Quarry/Bschhof_Quarry_Schretzmair_Quarry_ENMAP01-____L2A-DT0000001702_20220717T110358Z_005_V010505_20260205T022527Z.nc
 PNG  : /Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Bschhof_Quarry_Schretzmair_Quarry/Bschhof_Quarry_Schretzmair_Quarry_ENMAP01-____L2A-DT0000001702_20220717T110358Z_005_V010505_20260205T022527Z_rgb.png
Holzbachacker_clay_pit: clipped None? False
 NetCDF: /Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Holzbachacker_clay_pit/Holzbachacker_clay_pit_ENMAP01-____L2A-DT0000020106_20230525T105309Z_022_V010505_20260205T022344Z.nc
 PNG  : /Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Holzbachacker_clay_pit/Holzbachacker_clay_pit_ENMAP01-____L2A-DT0000020106_20230525T105309Z_022_V010505_20260205T022344Z_rgb.png
La_Sanguinde_mine: clipped None? False
 NetCDF: /Volumes/T7 Shield/sphalerite_output/Cuprite Mines/La_Sanguinde_m

In [60]:
from pathlib import Path
import shutil

def collect_mines(mineral_root: Path, mines_subdir: str, out_dir: Path, dry_run: bool = False):
    mines_dir = mineral_root / mines_subdir
    assert mines_dir.exists(), f"Missing mines dir: {mines_dir}"

    png_out = out_dir / f"{mineral_root.name}_png"
    nc_out  = out_dir / f"{mineral_root.name}_netcdf"
    png_out.mkdir(parents=True, exist_ok=True)
    nc_out.mkdir(parents=True, exist_ok=True)

    mine_folders = [p for p in mines_dir.iterdir() if p.is_dir()]
    print(f"Found {len(mine_folders)} mine folders under {mines_dir}")

    for mine in mine_folders:
        mine_name = mine.name

        pngs = sorted(mine.rglob("*.png"))
        ncs  = sorted(mine.rglob("*.nc"))

        if not pngs and not ncs:
            print(f"[SKIP] {mine_name}: no png/nc")
            continue

        # If you expect exactly 1 png + 1 nc per mine, enforce it:
        # assert len(pngs)==1 and len(ncs)==1, f"{mine_name}: expected 1 png and 1 nc"

        for src in pngs:
            dst_name = f"{mine_name}__{src.name}"
            dst = png_out / dst_name
            if dry_run:
                print("[DRY] COPY", src, "->", dst)
            else:
                shutil.copy2(src, dst)

        for src in ncs:
            dst_name = f"{mine_name}__{src.name}"
            dst = nc_out / dst_name
            if dry_run:
                print("[DRY] COPY", src, "->", dst)
            else:
                shutil.copy2(src, dst)

    print("Done.")
    print("PNG:", png_out)
    print("NC :", nc_out)

if __name__ == "__main__":
    # Example usage for magnesite:
    mineral_root = Path('/Volumes/T7 Shield/cuprite_output')              # change
    mines_subdir = "Cuprite Mines"                     # change
    out_dir      = Path("dataset_staging")               # change
    collect_mines(mineral_root, mines_subdir, out_dir, dry_run=False)

Found 240 mine folders under /Volumes/T7 Shield/cuprite_output/Cuprite Mines
Done.
PNG: dataset_staging/cuprite_output_png
NC : dataset_staging/cuprite_output_netcdf


In [70]:
import glob
glob.glob('/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/*')

['/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Bschhof_Quarry_Schretzmair_Quarry',
 '/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Holzbachacker_clay_pit',
 '/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/La_Sanguinde_mine',
 '/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Guelph_Dolime_Quarry',
 '/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/Thomaston_Dam_railroad_cut',
 '/Volumes/T7 Shield/sphalerite_output/Cuprite Mines/O__G_Southbury_Quarry_Silliman_Quarry_O__G_No_2_Quarry']

### ------------------------------------------------------------------------------------------
### Run Random Forests
### ------------------------------------------------------------------------------------------


In [ ]:
nc_root   = Path("Magnesite Mines")          # folders per mine
mask_dir  = Path("masks_png")  # your mask folder

def scene_key(name: str) -> str:
    # Extract the unique key from the filename
    m = re.search(r"(.*?ENMAP01-.*?_V010502)", name)
    if not m:
        raise ValueError(f"Cannot extract scene key from: {name}")
    return m.group(1)

# Index masks by key
mask_by_key = {}
for mp in mask_dir.glob("*_mask.png"):
    k = scene_key(mp.name)
    mask_by_key[k] = mp

pairs = []
missing = []

# Find all nc files in subfolders
for nc in nc_root.glob("*/*.nc"):
    k = scene_key(nc.name)
    mp = mask_by_key.get(k)
    if mp is None:
        missing.append(nc)
        continue
    pairs.append((k, mp, nc))

print("pairs:", len(pairs))
print("missing masks:", len(missing))

In [ ]:

ds = xr.open_dataset('ENMAP_Magnesite/ENMAP.HSI.L2A/ENMAP01-____L2A-DT0000001849_20220720T104311Z_020_V010502_20251103T212534Z.ZIP')
ds[0].plot()

In [ ]:
import xarray as xr
import imageio.v2 as imageio

def check_pair(mask_path, nc_path, var="reflectance"):
    ds = xr.open_dataset(nc_path, engine="netcdf4")
    cube = ds[var]
    mask = imageio.imread(mask_path)

    Hm, Wm = mask.shape[:2]
    Hy, Wy = cube.sizes[cube.dims[-2]], cube.sizes[cube.dims[-1]]

    assert (Hm, Wm) == (Hy, Wy), f"Shape mismatch mask {(Hm,Wm)} vs cube {(Hy,Wy)}"
    return True

for k, mp, nc in pairs[:3]:
    check_pair(mp, nc)
print("alignment ok (sample)")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import imageio.v2 as imageio
import xarray as xr

keys = [k for k,_,_ in pairs]
train_k, test_k = train_test_split(keys, test_size=0.3, random_state=42)

train_pairs = [p for p in pairs if p[0] in train_k]
test_pairs  = [p for p in pairs if p[0] in test_k]

def pixels_from_pairs(pairs, var="reflectance", max_bg_per_image=5000):
    Xs, ys = [], []

    for k, mp, nc in pairs:
        ds = xr.open_dataset(nc, engine="netcdf4")
        cube = ds[var].values  # (B, y, x)
        mask = (imageio.imread(mp) > 0)  # (y, x) boolean

        B, H, W = cube.shape
        X = cube.reshape(B, -1).T      # (N, B)
        y = mask.reshape(-1).astype(np.uint8)

        good = np.isfinite(X).all(axis=1)
        X, y = X[good], y[good]

        # class balancing: subsample background so RF doesn't learn "always 0"
        bg_idx = np.where(y == 0)[0]
        fg_idx = np.where(y == 1)[0]

        if len(bg_idx) > max_bg_per_image:
            bg_idx = np.random.choice(bg_idx, size=max_bg_per_image, replace=False)

        keep = np.concatenate([bg_idx, fg_idx])
        Xs.append(X[keep])
        ys.append(y[keep])

    return np.vstack(Xs), np.hstack(ys)

X_train, y_train = pixels_from_pairs(train_pairs)
X_test,  y_test  = pixels_from_pairs(test_pairs)

rf = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(classification_report(y_test, pred, target_names=["bg","magnesite"]))

# random 

In [ ]:


keys = [k for k,_,_ in pairs]
train_k, test_k = train_test_split(keys, test_size=0.3, random_state=42)

train_pairs = [p for p in pairs if p[0] in train_k]
test_pairs  = [p for p in pairs if p[0] in test_k]

def pixels_from_pairs(pairs, var="reflectance", max_bg_per_image=5000):
    Xs, ys = [], []

    for k, mp, nc in pairs:
        ds = xr.open_dataset(nc, engine="netcdf4")
        cube = ds[var].values  # (B, y, x)
        mask = (imageio.imread(mp) > 0)  # (y, x) boolean

        B, H, W = cube.shape
        X = cube.reshape(B, -1).T      # (N, B)
        y = mask.reshape(-1).astype(np.uint8)

        good = np.isfinite(X).all(axis=1)
        X, y = X[good], y[good]

        # class balancing: subsample background so RF doesn't learn "always 0"
        bg_idx = np.where(y == 0)[0]
        fg_idx = np.where(y == 1)[0]

        if len(bg_idx) > max_bg_per_image:
            bg_idx = np.random.choice(bg_idx, size=max_bg_per_image, replace=False)

        keep = np.concatenate([bg_idx, fg_idx])
        Xs.append(X[keep])
        ys.append(y[keep])

    return np.vstack(Xs), np.hstack(ys)

X_train, y_train = pixels_from_pairs(train_pairs)
X_test,  y_test  = pixels_from_pairs(test_pairs)

rf = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(classification_report(y_test, pred, target_names=["bg","magnesite"]))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

def evaluate_split(model, X, y, split_name=""):
    y_pred = model.predict(X)

    print(f"\n=== {split_name} classification report ===")
    print(classification_report(
        y, y_pred, target_names=["bg", "magnesite"]
    ))

    cm = confusion_matrix(y, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["bg_true", "magnesite_true"],
        columns=["bg_pred", "magnesite_pred"]
    )

    print(f"\n=== {split_name} confusion matrix ===")
    print(cm_df)

    return cm

In [ ]:
cm_train = evaluate_split(rf, X_train, y_train, "TRAIN")
cm_test  = evaluate_split(rf, X_test,  y_test,  "TEST")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import imageio.v2 as imageio
import xarray as xr

import numpy as np
import xarray as xr
import imageio.v2 as imageio
import matplotlib.pyplot as plt

def visualize_test_image(pair, model, bands=(40, 20, 10), var="reflectance"):
    k, mp, nc = pair

    ds = xr.open_dataset(nc, engine="netcdf4")
    cube = ds[var].values  # (B,H,W)
    mask_gt = imageio.imread(mp) > 0

    B, H, W = cube.shape
    X = cube.reshape(B, -1).T  # (Npix, B)

    good = np.isfinite(X).all(axis=1)

    n_good = int(good.sum())
    if n_good == 0:
        print(f"[SKIP] {k}: 0 finite pixels. Cube stats: "
              f"nan={np.isnan(cube).mean():.3f}, inf={np.isinf(cube).mean():.3f}, "
              f"min={np.nanmin(cube):.4g}, max={np.nanmax(cube):.4g}")
        return

    pred_flat = np.zeros(H * W, dtype=np.uint8)
    pred_flat[good] = model.predict(X[good])
    pred = pred_flat.reshape(H, W)

    # RGB display
    bb = [b for b in bands if b < B]
    rgb = np.stack([cube[b] for b in bb], axis=-1)
    rgb = rgb / np.nanpercentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb); axs[0].set_title("RGB")
    axs[1].imshow(rgb); axs[1].imshow(mask_gt, alpha=0.4); axs[1].set_title("GT mask")
    axs[2].imshow(rgb); axs[2].imshow(pred, alpha=0.4); axs[2].set_title("RF prediction")
    for ax in axs: ax.axis("off")
    plt.suptitle(k)
    plt.show()

In [ ]:
for pair in test_pairs[:5]:
    visualize_test_image(pair, rf)

# Spectral GPT 

In [ ]:
import xarray as xr
import numpy as np

nc_path = "Brgl_mine_Brglkopf_mine/Brgl_mine_Brglkopf_mine_ENMAP01-____L2A-DT0000001849_20220720T104311Z_020_V010502_20251103T212534Z_chip128.nc"  # <- edit
ds = xr.open_dataset(nc_path, engine="netcdf4")

cube = ds["reflectance"].values.astype(np.float32)  # (B,H,W)
B, H, W = cube.shape
assert (H, W) == (128, 128), cube.shape

# Replace NaNs/Infs so the model doesn’t choke
cube = np.nan_to_num(cube, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
# keep only bands divisible by 3
B3 = (B // 3) * 3
cube3 = cube[:B3]                           # (B3,128,128)
G = B3 // 3
cube_g3 = cube3.reshape(G, 3, H, W)         # (G,3,128,128)

In [ ]:
cube_g3.shape

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- pick ONE model definition ----
# Try tensor ViT first (often expects general spectral input)
import sys
sys.path.append('third_party/spectralgpt/') 
import models_vit_tensor as m

# Common constructor names in their repo: mae_vit_* / vit_* depending on file
# If this name doesn't exist in your local clone, list dir(m) and pick the right one.
model = m.vit_base_patch8()  # <- you may need to change to a valid constructor in that file

# ---- load checkpoint if you have it ----
ckpt_path = "PATH/TO/CHECKPOINT.pth"  # <- edit (or comment out if you don't have weights)
ckpt = torch.load(ckpt_path, map_location="cpu")
state = ckpt.get("model", ckpt)       # common pattern: sometimes stored under 'model'
model.load_state_dict(state, strict=False)

model.eval().to(device)

# ---- build input ----
# We feed ONE grouped-3 image at a time: (N,3,128,128)
x = torch.from_numpy(cube_g3).to(device)   # (G,3,128,128)

# ---- extract representations ----
with torch.no_grad():
    if hasattr(model, "forward_features"):
        feats = model.forward_features(x)
    else:
        feats = model(x)

# feats is your representation tensor (shape depends on the model)
print(type(feats), getattr(feats, "shape", None))